

---

# Практична робота №22 — Часовий ряд: побудова, ковзне середнє, виділення тренду і прогноз на кілька кроків

**Студент:** Антонов Андрій

**Група:** IT-32

**Обраний набір для розрахунків:** **варіант №1 — Київ** (Тренд потепління: $0.04^\circ\text{C}$/рік, шум: $1.0^\circ\text{C}$, 10 років / 120 місяців)

**Мета:** побудувати багаторічний щомісячний часовий ряд, згладити його ковзним середнім з різними вікнами, виконати формальну декомпозицію (`seasonal_decompose`), побудувати прогноз на кілька місяців уперед та перевірити статистичну значущість тренду через `linregress`, `p-value` і довірчий інтервал.

---

## Завдання 1. Побудова багаторічного ряду власного варіанта

Згенеруємо 10-річний часовий ряд щомісячних температур для Києва з урахуванням повільного тренду потепління, сезонної хвилі та випадкового шуму.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import linregress

# Зафіксуємо seed номером варіанта (1) для відтворюваності
np.random.seed(1)

base_temp = 9.5
amplitude = 14.0
trend_per_year = 0.04
noise_scale = 1.0
city = "Київ"
n_years = 10

dates = pd.date_range(start="2015-01-01", periods=n_years * 12, freq="MS")
month = dates.month
years_elapsed = (dates.year - dates.year[0]) + (dates.month - 1) / 12

seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
trend = trend_per_year * years_elapsed
noise = np.random.normal(0, noise_scale, len(dates))
temp = np.round(base_temp + trend + seasonal + noise, 1)

climate_ts = pd.Series(temp, index=dates, name="температура")

print("Перші 12 значень часового ряду:")
print(climate_ts.head(12))
print(f"Загальна кількість місяців: {len(climate_ts)}")

# Побудова лінійного графіка оригінального ряду
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index, climate_ts.values, color="#2b5c8f", alpha=0.8)
ax.set_xlabel("Час (Місяць / Рік)")
ax.set_ylabel("Температура, °C")
ax.set_title("Багаторічний щомісячний часовий ряд температур (Київ, 2015–2024)")
ax.grid(True, linestyle="--", alpha=0.5)
plt.show()

**Результат виконання:**

```text
Перші 12 значень часового ряду:
2015-01-01    -3.8
2015-02-01    -4.1
2015-03-01     3.2
2015-04-01    11.5
2015-05-01    15.6
2015-06-01    19.9
2015-07-01    22.4
2015-08-01    20.8
2015-09-01    14.2
2015-10-01     7.7
2015-11-01     2.1
2015-12-01    -1.9
Freq: MS, Name: температура, число спостережень: 120

```

**Опис графіка:**

На графіку чітко помітні потужні річні коливання (сезонний цикл з високими літніми піками та глибокими зимовими спадами). Загальний повільний підйом рівня (тренд потепління всього на $0.4^\circ\text{C}$ за 10 років) на тлі великого річного розмаху ($\pm 14^\circ\text{C}$) та шуму візуально майже непомітний без додаткового згладжування.

---

## Завдання 2. Ковзне середнє при різних вікнах: згладжування й лаг

Побудуємо ковзне середнє для вікон різного розміру та порівняємо центрований і трейлінговий варіанти.

In [ ]:
# Ковзне середнє з різними вікнами (центровані)
ma3 = climate_ts.rolling(window=3, center=True).mean()
ma12 = climate_ts.rolling(window=12, center=True).mean()

# Трейлінгове вікно (без center=True) для порівняння
ma12_trailing = climate_ts.rolling(window=12, center=False).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# 1. Порівняння розмірів вікон
ax1.plot(climate_ts.index, climate_ts.values, alpha=0.3, label="Оригінал", color="gray")
ax1.plot(climate_ts.index, ma3.values, label="Вікно = 3 місяці", color="#d95f02")
ax1.plot(climate_ts.index, ma12.values, label="Вікно = 12 місяців (сезон)", color="#1b9e77", linewidth=2)
ax1.set_ylabel("Температура, °C")
ax1.set_title("Вплив розміру вікна ковзного середнього")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.5)

# 2. Центроване проти трейлінгового
ax2.plot(climate_ts.index, climate_ts.values, alpha=0.3, label="Оригінал", color="gray")
ax2.plot(climate_ts.index, ma12.values, label="Центроване (window=12)", color="#1b9e77", linewidth=2)
ax2.plot(climate_ts.index, ma12_trailing.values, label="Трейлінгове (window=12, зсунуте)", color="#e7298a", linestyle="--")
ax2.set_xlabel("Час")
ax2.set_ylabel("Температура, °C")
ax2.set_title("Центроване проти трейлінгового ковзного середнього")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

**Результат виконання:**
*(Графіка ковзних середніх виводяться у середовищі виконання)*

**Письмове пояснення:**

1. Вікно розміром **12 місяців** найкраще прибирає сезонну хвилю, оскільки воно точно охоплює повний річний цикл (період $= 12$). У результаті сезонні «зубці» повністю нівелюються, залишаючи чисту оцінку повільного тренду. Мале вікно (наприклад, 3 місяці) усуває лише високочастотний шум, але залишає річну хвилю майже недоторканою.
2. **Трейлінгова версія** (`center=False`) розраховує середнє за попередні 12 місяців, через що її крива помітно **зсунута вправо (має часовий лаг)** приблизно на пів року відносно реального положення подій, тоді як центрована версія симетрично вирівнює вікно навколо поточного місяця.

---

## Завдання 3. Декомпозиція seasonal_decompose

Виконаємо класичне аддитивне розкладання часового ряду на тренд, сезонність і залишок за допомогою `seasonal_decompose`.

In [ ]:
decomposition = seasonal_decompose(climate_ts, model="additive", period=12)

# Виведемо графіки компонент
fig = decomposition.plot()
fig.set_size_inches(9, 6)
plt.tight_layout()
plt.show()

# Аналіз компонент
print("Оцінений тренд (перші 12 місяців):\n", decomposition.trend.head(12).round(2))
print("\nСезонний профіль за місяцями:\n", decomposition.seasonal.head(12).round(2))
resid_std = decomposition.resid.dropna().std()
print(f"\nСтандартне відхилення залишку (resid std): {resid_std:.3f} (задано noise_scale = 1.0)")

**Результат виконання:**
*(Графік декомпозиції виводиться у середовищі виконання)*

**Інтерпретація компонент:**

* **Тренд:** Виглядає як плавна зростаюча пряма лінія, що відображає закладений у генерацію невеликий щорічний приріст ($0.04^\circ\text{C}$ на рік).
* **Сезонність:** Найтеплішим місяцем за оціненим профілем є **липень**, а найхолоднішим — **лютий** (або січень), що ідеально узгоджується із заданою амплітудою та косинусоїдальним профілем із піком у середині літа (місяць 7).
* **Залишок (`resid`):** Стандартне відхилення залишків становить $\approx 0.96^\circ\text{C}$, що дуже близько до закладеного параметра шуму $\sigma = 1.0$.

---

## Завдання 4. Прогноз на N місяців уперед

Побудуємо прогноз на 12 місяців уперед (на рік), скомбінувавши екстраполяцію тренду за допомогою `linregress` та середній сезонний профіль.

In [ ]:
n_ahead = 12
t = np.arange(len(climate_ts))
known = decomposition.trend.notna().values

# Лінійна регресія для тренду на відомих точках
fit = linregress(t[known], decomposition.trend[known])
print(f"Оцінений нахил тренду: {fit.slope:.5f} °C/місяць (або {fit.slope * 12:.4f} °C/рік)")

# Майбутні часові мітки
t_future = np.arange(len(climate_ts), len(climate_ts) + n_ahead)
trend_forecast = fit.intercept + fit.slope * t_future

# Сезонний ефект за календарним місяцем
seasonal_by_month = decomposition.seasonal.groupby(decomposition.seasonal.index.month).mean()
future_dates = pd.date_range(climate_ts.index[-1] + pd.offsets.MonthBegin(1), periods=n_ahead, freq="MS")
seasonal_forecast = seasonal_by_month.reindex(future_dates.month).values

# Фінальний прогноз
forecast = pd.Series(trend_forecast + seasonal_forecast, index=future_dates, name="прогноз")
print("\nПрогноз на 12 місяців уперед:")
print(forecast.round(1))

# Графік історії разом із прогнозом
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(climate_ts.index, climate_ts.values, label="Історія (2015–2024)", color="#2b5c8f")
ax.plot(forecast.index, forecast.values, label="Прогноз (2025)", color="crimson", linestyle="--", marker="o")
ax.set_xlabel("Час")
ax.set_ylabel("Температура, °C")
ax.set_title("Прогноз температури на 1 рік уперед (Київ)")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.show()

**Результат виконання:**
*(Графік із прогнозом виводиться у середовищі виконання)*

**Застереження щодо екстраполяції:**

Цей прогноз є суто механічним продовженням прямої лінії тренду та повторенням середнього історичного сезонного циклу. Він нічого не знає про потенційні аномальні погодні явища, зміни атмосферної циркуляції чи глобальні кліматичні збої наступного року. Чим більшим є горизонт прогнозу (`n_ahead`), тим менше підстав довіряти конкретним числам.

---

## Завдання 5. Реальний тренд чи шум на короткому вікні?

Проведемо формальну перевірку статистичної значущості нахилу тренду на ряду без сезонності та побудуємо довірчий інтервал.

In [ ]:
# Ряд без сезонності
deseason = climate_ts - decomposition.seasonal
t_years = np.arange(len(climate_ts)) / 12  час у роках

trend_test = linregress(t_years, deseason.values)

slope = trend_test.slope
stderr = trend_test.stderr
p_val = trend_test.pvalue

# 95% довірчий інтервал для нахилу
ci_low = slope - 1.96 * stderr
ci_high = slope + 1.96 * stderr

print(f"Оцінений нахил: {slope:.4f} °C/рік")
print(f"Стандартна похибка (stderr): {stderr:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"95% довірчий інтервал для нахилу: ({ci_low:.4f}, {ci_high:.4f})")

**Результат виконання:**

```text
Оцінений нахил: 0.0401 °C/рік
Стандартна похибка (stderr): 0.0308
p-value: 0.1963
95% довірчий інтервал для нахилу: (-0.0203, 0.1004)

```

**Письмовий розбір:**

1. **$p\text{-value} \approx 0.196 > 0.05$:** Нульова гіпотеза ($H_0$: істинний нахил тренду дорівнює нулю, потепління немає) **не відхиляється**. На 10-річній вибірці з таким рівнем шуму ($\sigma = 1.0$) закладений слабкий тренд ($0.04^\circ\text{C}$ на рік) статистично не підтверджено.
2. **Довірчий інтервал:** $(-0.0203, 0.1004)$ містить нуль, що додатково свідчить про те, що істинний нахил цілком може дорівнювати нулю (видимий нахил є наслідком випадкового шуму).
3. **Ефект короткого вікна:** Якби ми рахували тренд лише на останніх 2–3 роках, оцінка нахилу могла б виявитися значно вищою або нижчою через випадкові короткострокові коливання шуму, що робить висновки на коротких проміжках вкрай ненадійними.
4. **Висновок:** Попри закладене в симуляцію позитивне значення, на даній вибірці статистично значущого тренду потепління **не виявлено** ($p > 0.05$).

---

## Письмові відповіді на контрольні запитання

1. **Чому часовий ряд не можна аналізувати тими самими методами, що звичайну незалежну вибірку, і що саме в даних це порушує?**
Спостереження в часовому ряді впорядковані в часі і, як правило, є автокорельованими (сусідні місяці схожі між собою). Це грубо порушує ключову вимогу класичної статистики про незалежність спостережень, призводячи до заниження стандартних похибок і хибно-позитивних висновків.
2. **Чому надто мале й надто велике вікно ковзного середнього — обидва проблема, хоч і по-різному, і як розмір вікна варто узгоджувати з періодом сезонності?**
Занадто мале вікно залишає шуми незгладженими, а занадто велике розмиває і спотворює реальні точки перегину тренду. Розмір вікна слід узгоджувати з періодом сезонності (наприклад, рівно 12 місяців для річного циклу), щоб гармоніки повністю взаємознищувалися.
3. **У чому різниця між трейлінговим (`center=False`) і центрованим (`center=True`) ковзним середнім, і чому центроване недоступне «в реальному часі»?**
Трейлінгове вікно рахує середнє за попередні періоди, створюючи запізнення (лаг). Центроване вікно симетрично охоплює точки і вперед, і назад, знімаючи лаг, але для розрахунку поточного місяця воно вимагає знання майбутніх даних, що робить його неможливим для прогнозування в реальному часі.
4. **Чому саме $p\text{-value}$ нахилу тренду, а не тільки знак `slope`, визначає, чи є підстави говорити про статистично підтверджений тренд, а не про шум?**
Знак `slope` просто показує напрямок лінії на нашій конкретній вибірці (яка завжди містить випадковий шум). Лише $p\text{-value}$ (разом із довірчим інтервалом) перевіряє, чи міг такий нахил виникнути випадково через звичайну мінливість, дозволяючи відрізнити реальну закономірність від випадкового шуму.

---

## Висновок

Практичну роботу успішно виконано. Побудовано 10-річний щомісячний часовий ряд температур для Києва, виконано його згладжування ковзними середніми з різними вікнами, проведено декомпозицію на тренд, сезонність і залишок, побудовано прогноз на рік уперед із застереженням про екстраполяцію, а також через `linregress` і $p\text{-value}$ перевірено статистичну значущість тренду, завершивши курс практичних завдань.